# S8 · Lab 04 — Interdependencia: el reto del ajuste manual (y el nacimiento del Backprop)

## Narrativa
Vale. Ya sabemos que necesitamos:
- capas,
- activaciones.

Pero ahora aparece el problema real:

> “¿Cómo ajustamos los pesos de este sistema?”

Este laboratorio te obliga a intentarlo **a mano**… para que compruebes por qué es inviable.

---

## Qué vas a producir
1) Un mini-lote con varios ejemplos: verás que un cambio “arregla uno y rompe otros”.  
2) Un mini-juego: tienes 3 intentos para bajar el error tocando **un solo peso**.  
3) La conclusión inevitable: necesitamos un método global donde el error “fluya hacia atrás”.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    h = relu(X @ W1 + b1)
    y_hat = sigmoid(h @ W2 + b2)
    return h, y_hat

def bce(y, y_hat, eps=1e-9):
    y_hat = np.clip(y_hat, eps, 1-eps)
    return -(y*np.log(y_hat) + (1-y)*np.log(1-y_hat))

In [ ]:
# 1) Mini-lote (varios puntos) para ver el efecto 'arreglo 1, rompo 3'
X = np.array([
    [ 0.7, -1.2],
    [ 1.1,  0.2],
    [-0.3,  0.9],
    [ 0.0, -0.6],
    [-1.0, -0.4],
    [ 0.4,  1.3],
])

y = np.array([[1],[1],[0],[0],[0],[1]], dtype=float)

# Red 2->3->1
W1 = np.array([[0.2, -0.1, 0.05],
               [0.3,  0.4, -0.2]])
b1 = np.zeros((1, 3))

W2 = np.array([[0.6],
               [-0.4],
               [0.2]])
b2 = np.zeros((1, 1))

_, y_hat = forward(X, W1, b1, W2, b2)
loss_vec = bce(y, y_hat)
loss0 = float(np.mean(loss_vec))

print("y_hat:", np.round(y_hat.T, 3))
print("loss por punto:", np.round(loss_vec.T, 3))
print("loss media:", round(loss0, 3))

In [ ]:
# 2) Elige un objetivo: intenta mejorar SOLO el punto 0
target = 0
print("Punto objetivo:", target, "X=", X[target], "y=", int(y[target,0]))

## Mini-juego: 3 intentos para bajar el error tocando un solo peso

Reglas:
- Solo puedes tocar **W1[0,0]** (un único peso de la primera capa).
- Tienes **3 intentos**.
- Objetivo: bajar la **loss media** por debajo de **0.10** (muy difícil a propósito).

Lo importante no es ganar.
Lo importante es ver el patrón: *arreglas un caso y rompes otros*.


In [ ]:
def evaluate(W1_current):
    _, yhat = forward(X, W1_current, b1, W2, b2)
    lv = bce(y, yhat)
    return float(np.mean(lv)), lv, yhat

W1_current = W1.copy()
best = evaluate(W1_current)[0]

for attempt in range(1, 4):
    print("\n--- Intento", attempt, "---")
    print("Peso actual W1[0,0] =", round(float(W1_current[0,0]), 4))
    s = input("Nuevo valor para W1[0,0] (ENTER para probar +0.15): ").strip()
    if s == "":
        W1_current[0,0] = W1_current[0,0] + 0.15
    else:
        W1_current[0,0] = float(s)

    mean_loss, lv, yhat = evaluate(W1_current)
    mean_loss_history.append(mean_loss)
    w_history.append(float(W1_current[0,0]))
    print("y_hat:", np.round(yhat.T, 3))
    print("loss por punto:", np.round(lv.T, 3))
    print("loss media:", round(mean_loss, 3))

    if mean_loss < best:
        best = mean_loss
        print("✅ Mejoras el global (por ahora).")
    else:
        print("⚠️ No mejoras el global o lo empeoras.")

    # Destaca el objetivo vs los demás
    print("Punto objetivo loss:", round(float(lv[target]), 3), "| antes era:", round(float(loss_vec[target]), 3))

    # --- DETECTOR AUTOMÁTICO DE SABOTAJE ---
    # Calculamos la diferencia de pérdida por cada punto individual respecto al estado inicial
    diff_per_point = lv - loss_vec

    # Encontramos los índices que más han empeorado (mayor aumento de loss)
    peores_indices = np.argsort(diff_per_point.flatten())[::-1][:3]

    print("-" * 50)
    print("🚨 INFORME DE DAÑOS COLATERALES 🚨")
    print("-" * 50)

    for i in peores_indices:
        empeoramiento = float(diff_per_point.flatten()[i])
        if empeoramiento > 0:
            print(f"Fila {int(i)}: La pérdida HA SUBIDO en {empeoramiento:.4f}")
        else:
            print(f"Fila {int(i)}: (Sin daños significativos o mejora leve)")

    print("-" * 50)
    print("💡 REFLEXIÓN: ¿Ves cómo al intentar ayudar al 'Punto Objetivo',")
    print("has 'saboteado' el aprendizaje de otras filas?")
    # --- VISUAL: Top-3 daños colaterales (barras) ---
    diffs = diff_per_point.flatten()[peores_indices].astype(float)
    labels = [str(int(i)) for i in peores_indices]

    plt.figure()
    plt.bar(labels, diffs)
    plt.axhline(0)
    plt.title("Top-3 cambios de pérdida vs estado inicial (por fila)")
    plt.xlabel("Fila")
    plt.ylabel("Δ loss (actual - inicial)")
    plt.show()



    # --- VISUAL: Antes vs Después (loss por fila) ---
    plt.figure()
    x_pos = np.arange(len(lv))
    plt.bar(x_pos - 0.2, loss_vec.flatten(), width=0.4, label="Inicial")
    plt.bar(x_pos + 0.2, lv.flatten(), width=0.4, label="Actual")
    plt.title("Loss por fila: inicial vs actual (mira el daño colateral)")
    plt.xlabel("Fila")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    if mean_loss <= 0.10:
        print("🎯 ¡Objetivo logrado! (raro)"); break

## Resumen visual final
Aunque solo hayas tocado **un peso**, el sistema ha cambiado globalmente.

Mira cómo evoluciona la loss media por intento.


In [ ]:
if len(mean_loss_history) > 0:
    plt.figure()
    plt.plot(range(1, len(mean_loss_history)+1), mean_loss_history, marker="o")
    plt.title("Evolución de la loss media por intento (ajuste manual)")
    plt.xlabel("Intento")
    plt.ylabel("Loss media")
    plt.show()

    plt.figure()
    plt.plot(range(1, len(w_history)+1), w_history, marker="o")
    plt.title("Valor probado para W1[0,0] por intento")
    plt.xlabel("Intento")
    plt.ylabel("W1[0,0]")
    plt.show()
else:
    print("No hay historial (¿no ejecutaste el bucle?)")

In [ ]:
# Guardaremos un historial para visualizar el aprendizaje (o sabotaje) intento a intento
mean_loss_history = []
w_history = []

## Momento “¡Aja!”

1) ¿Has notado que, al mejorar el punto objetivo, otros empeoran?  
2) ¿Por qué ajustar “a ojo” un peso temprano tiene efectos globales?  
3) ¿Qué información necesitaría una capa temprana para saber si ha ayudado o perjudicado?

📌 Frase de maestro (para el aula)
> “Habéis tocado un solo peso para arreglar una predicción y habéis roto tres más.  
> Por eso no ajustamos redes a mano. Por eso inventamos el Backpropagation.”



## Preguntas de cierre (responde aquí)

1) Describe un ejemplo concreto de “arreglé uno y rompí otros” observado en tu intento.  
2) ¿Por qué este laboratorio hace inevitable el entrenamiento global guiado por el error?  
3) ¿Qué riesgo aparece si optimizas un error mal definido (aunque baje)?

